In [1]:
# !pip install --upgrade pip
# !python3.7 -m pip install rank_bm25
# !pip install sentence-transformers==2.2.2 transformers==4.28.1 --no-cache-dir
# !pip install faiss-cpu
# !pip install nltk
# !python3.7 -m pip install stanza
# !pip install --upgrade typing_extensions==4.5.0

In [2]:
import pandas as pd
import numpy as np
import re
import pyspark
from pyspark.sql import SparkSession
import shutil
import json
from rank_bm25 import BM25Okapi

import nltk
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
stopwords_set = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /mnt/vocwork3/work/eee_W_4030
[nltk_data]     962/asn3822444_20/asn3822445_1/work/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to /mnt/voc
[nltk_data]     work3/work/eee_W_4030962/asn3822444_20/asn3822445_1/wo
[nltk_data]     rk/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /mnt/vocwork3/work/eee_W_
[nltk_data]     4030962/asn3822444_20/asn3822445_1/work/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# pyspark works best with java8 
# set JAVA_HOME enviroment variable to java8 path 
%env JAVA_HOME = /usr/lib/jvm/java-8-openjdk-amd64

env: JAVA_HOME=/usr/lib/jvm/java-8-openjdk-amd64


In [4]:
spark = SparkSession.builder.getOrCreate()
# sc = pyspark.SparkContext()

## Read Dataset and Preprocessing Data

In [5]:
# Small subset of training dataset - 30% of original data
# Full dataset: s3://nina-rag-project/wiki_movie_plots_deduped.csv

df = spark.read.csv("data/movie_subset.csv", header=True, multiLine=True, escape="\"", quote="\"")
# columns = [
#     "Release Year", "Title", "Origin/Ethnicity", "Director", "Cast", 
#     "Genre", "Wiki Page", "Plot"
# ]
# df = df.toDF(*columns)
# df = spark.read.csv("data/wiki_movie_plots_kaggle.csv", header=True, multiLine=True, escape="\"", quote="\"")

df.printSchema()

root
 |-- Release Year: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Origin/Ethnicity: string (nullable = true)
 |-- Director: string (nullable = true)
 |-- Cast: string (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Wiki Page: string (nullable = true)
 |-- Plot: string (nullable = true)
 |-- plot_length: string (nullable = true)



In [6]:
print(df.count()) 

10466


In [7]:
from pyspark.sql.functions import col, when, count, trim

missing_counts = df.select([
    count(when(col(c).isNull() | (trim(col(c)) == ""), c)).alias(c)
    for c in df.columns
])

missing_counts.show()

+------------+-----+----------------+--------+----+-----+---------+----+-----------+
|Release Year|Title|Origin/Ethnicity|Director|Cast|Genre|Wiki Page|Plot|plot_length|
+------------+-----+----------------+--------+----+-----+---------+----+-----------+
|           0|    0|               0|       0|   0|    5|        0|   0|          0|
+------------+-----+----------------+--------+----+-----+---------+----+-----------+



In [8]:
from pyspark.sql.functions import length

df = df.withColumn("plot_length", length(df["Plot"]))
df.select("plot_length").describe().show()

+-------+------------------+
|summary|       plot_length|
+-------+------------------+
|  count|             10466|
|   mean|2156.6063443531434|
| stddev|1762.6956931513862|
|    min|                15|
|    max|             29442|
+-------+------------------+



In [9]:
def build_kv_pair(row):
    key = f"{row['Title']}_{row['Release Year']}"
    value = f"Movie: {row['Title']}\nOrigin: {row['Origin/Ethnicity']}\nGenre: {row['Genre']}\nDirector: {row['Director']}\nCast: {row['Cast']}\nPlot: {row['Plot']}"
    return (key, value)

movie_rdd = df.fillna("unknown").select("Title", "Release Year", 
                                        "Origin/Ethnicity", "Genre", 
                                        "Director", "Cast", 
                                        "Plot").rdd.map(build_kv_pair)

citation_pattern = re.compile(r'\[\d+\]')
url_pattern = re.compile(r'https?://\S+')
ws_pattern = re.compile(r'\s+')

def clean(data):
    key, text = data
    text = citation_pattern.sub('', text)
    text = text.replace("\\'", "'")
    text = url_pattern.sub('', text)
    text = ws_pattern.sub(' ', text).strip()
    return (key, text)
movie_rdd = movie_rdd.map(clean)

lengths_rdd = movie_rdd.map(lambda x: len(x[1]))
length_stats = lengths_rdd.stats()
print(length_stats)

lengths = lengths_rdd.collect()
percentiles = np.percentile(lengths, [25, 50, 75, 90, 95, 99])
print("25th, 50th, 75th, 90th, 95th, 99th percentiles:", percentiles)

(count: 10466, mean: 2294.7804318746307, stdev: 1767.8666132550372, max: 29496.0, min: 121.0)
25th, 50th, 75th, 90th, 95th, 99th percentiles: [ 856.  1795.5 3510.  4638.  5388.  7401. ]


In [10]:
# Set this based on combined text columns (value in rdd)
use_small_dataset = True
MIN_CHARS = 200 if use_small_dataset else 400
MAX_CHARS = 2500 if use_small_dataset else 3000

def truncate_and_filter_by_length(data):
    key, text = data
    text = text[:MAX_CHARS]
    if len(text) >= MIN_CHARS:
        return (key, text)
    else:
        return None

movie_rdd = movie_rdd.map(truncate_and_filter_by_length).filter(lambda x: x is not None)

In [11]:
print(f'The dataset after cleaning: {movie_rdd.count()} rows left (keep {round(movie_rdd.count()*100/df.count(), 2)}%)')

The dataset after cleaning: 10399 rows left (keep 99.36%)


In [12]:
movie_rdd.take(1)

[('The Day the Earth Stood Still_1951',
  'Movie: The Day the Earth Stood Still Origin: American Genre: science fiction Director: Robert Wise Cast: Michael Rennie, Patricia Neal Plot: When a flying saucer lands in Washington, D.C., the Army quickly surrounds it. A humanoid (Michael Rennie) emerges, announcing that he has come in peace. When he unexpectedly opens a small device, he is shot by a nervous soldier. A tall robot emerges from the saucer and quickly disintegrates the soldiers\' weapons. The alien orders the robot, Gort, to stop. He explains that the now-broken device was a gift for the President which would have enabled him "to study life on the other planets". The alien, Klaatu, is taken to Walter Reed Hospital. After surgery, he uses a salve to quickly heal his wound. Meanwhile, the Army is unable to enter the saucer; Gort stands outside, silent and unmoving. Klaatu tells the President\'s secretary, Mr. Harley (Frank Conroy), that he has a message that must be delivered to a

In [13]:
shutil.rmtree("data/movie_cleanedtexts.txt", ignore_errors=True)
movie_rdd.map(lambda x: f"{x[0]}\t{x[1]}").saveAsTextFile("data/movie_cleanedtexts.txt")

## Embedding with SentenceTransformer (SBERT)

In [14]:
from sentence_transformers import SentenceTransformer

In [15]:
def embed_partition(partition):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    for key, text in partition:
        yield (key, model.encode(text).tolist())

embedded_rdd = movie_rdd.mapPartitions(embed_partition)

In [16]:
embedded_rdd.take(1)

[('The Day the Earth Stood Still_1951',
  [-0.06717679649591446,
   0.04620165377855301,
   0.009534728713333607,
   -0.029726989567279816,
   0.01261981576681137,
   -0.02096189372241497,
   -0.021280862390995026,
   0.014371505007147789,
   0.02689589187502861,
   -0.03206678479909897,
   0.005169092211872339,
   -0.030190274119377136,
   -0.031659919768571854,
   0.0456189326941967,
   0.008380661718547344,
   -0.02258516289293766,
   -0.06040128320455551,
   -0.04317226633429527,
   -0.047506656497716904,
   0.024721916764974594,
   -0.07065072655677795,
   0.09857954829931259,
   0.012993874959647655,
   0.0761454850435257,
   -0.07936809957027435,
   0.061752159148454666,
   0.04772007837891579,
   0.006093935575336218,
   -0.04601333290338516,
   -0.03287418931722641,
   0.027601204812526703,
   -0.0027153033297508955,
   -0.08085108548402786,
   0.03183826804161072,
   -0.015403549186885357,
   0.03297685459256172,
   0.052466556429862976,
   -0.03904375061392784,
   -0.0283862

In [17]:
shutil.rmtree("data/movie_embeddings.txt", ignore_errors=True)
embedded_rdd.map(lambda x: f"{x[0]}\t{','.join(map(str, x[1]))}").saveAsTextFile("data/movie_embeddings.txt")

## Collect embedding files into one TSV

In [18]:
import glob
with open("movie_embeddings.tsv", "w") as outfile:
    for part in sorted(glob.glob("data/movie_embeddings.txt/part-*")):
        with open(part) as infile:
            outfile.write(infile.read())

titles = []
vectors = []

with open("movie_embeddings.tsv", "r") as f:
    for line in f:
        title, vector_str = line.strip().split("\t")
        vector = list(map(float, vector_str.split(",")))
        titles.append(title)
        vectors.append(vector)

embeddings = np.array(vectors).astype("float32")

## Retrieval System (Local on master node, not Spark)

### Dense Retrieval using Faiss

In [19]:
import faiss

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, "movie_rag_index.faiss")

with open("movie_keys.json", "w") as f:
    json.dump(titles, f)

In [20]:
title_to_text = {}
with open("data/movie_cleanedtexts.txt/part-00000", "r") as f:
    for line in f:
        key, val = line.strip().split("\t", 1)
        title_to_text[key] = val

### Sparse Retrieval using BM25

In [21]:
plots_filtered = list(title_to_text.values())
tokenized_corpus = [word_tokenize(doc.lower()) for doc in plots_filtered]
bm25 = BM25Okapi(tokenized_corpus)

In [22]:
def sparse_retrieve(query, k=5):
    tokenized_query = word_tokenize(query.lower())
    scores = bm25.get_scores(tokenized_query)
    top_k_indices = np.argsort(scores)[::-1][:k]
    return [plots_filtered[i] for i in top_k_indices], top_k_indices

### Hybrid Retrieval

In [23]:
import stanza
stanza.download("en")
nlp = stanza.Pipeline("en", processors="tokenize,pos,lemma")

2025-04-29 14:25:20 INFO: Downloading default packages for language: en (English) ...
2025-04-29 14:25:22 INFO: File exists: /mnt/vocwork3/work/eee_W_4030962/asn3822444_20/asn3822445_1/work/stanza_resources/en/default.zip
2025-04-29 14:25:31 INFO: Finished downloading models and saved to /mnt/vocwork3/work/eee_W_4030962/asn3822444_20/asn3822445_1/work/stanza_resources.
2025-04-29 14:25:31 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


2025-04-29 14:25:31 INFO: Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |

2025-04-29 14:25:31 INFO: Using device: cpu
2025-04-29 14:25:31 INFO: Loading: tokenize
2025-04-29 14:25:31 INFO: Loading: pos
2025-04-29 14:25:32 INFO: Loading: lemma
2025-04-29 14:25:32 INFO: Done loading processors!


In [24]:
genre_mapping = {
    "sci-fi": "science fiction",
    "scifi": "science fiction",
    "sci fi": "science fiction",
    "science fiction": "science fiction",
    "romcom": "romantic comedy",
    "bio": "biography",
    "doc": "documentary",
    "anime": "animated",
    "thriller": "thriller",
    "horror": "horror",
    "action": "action",
    "comedy": "comedy",
    "drama": "drama",
    "mystery": "mystery",
    "fantasy": "fantasy",
    "adventure": "adventure",
    "crime": "crime",
    "war": "war",
    "romance": "romance",
    "family": "family",
    "history": "history",
}

def detect_genres(query):
    query_lower = query.lower()
    detected = []
    for variant, canonical in genre_mapping.items():
        if variant in query_lower:
            detected.append(canonical)
    return list(set(detected))

query = "Suggest a science fiction movie, in which aliens come to Earth in 1990s"
print(detect_genres(query))

target_pos = {"NOUN", "PROPN", "ADJ", "NUM"}

def extract_keywords(text):
    doc = nlp(text)

    common_movie_words = {
        "movie", "film", "story", "show", "watch", "scene", "actor", "actress",
        "director", "character", "plot", "cast", "version", "remake", 
        "see", "watching", "episode", "cinema"
    }
    
    all_stopwords = stopwords_set.union(common_movie_words)

    keywords = []
    for sentence in doc.sentences:
        for word in sentence.words:
#             print(f"{word.text}\t{word.lemma}\t{word.upos}")
            if word.upos in target_pos and word.lemma.lower() not in all_stopwords:
                keywords.append(word.lemma.lower())

    return keywords


['science fiction']


In [25]:
def hybrid_retrieve(query, k=5, dense_weight=0.5, sparse_weight=0.5, boost_genre=True, boost_title=True):
    detected_genre = detect_genres(query)
    print("Detected genres:", detected_genre)

    keywords = extract_keywords(query)
    print("Extracted keywords:", keywords)
    processed_query = " ".join(keywords)

    # Encode query with SBERT
    model = SentenceTransformer('all-MiniLM-L6-v2')
    dense_vector = model.encode([processed_query]).astype("float32")
    D, I = index.search(dense_vector, k=10)
    dense_indices = I[0]

    # Sparse retrieval (BM25)
    _, sparse_indices = sparse_retrieve(processed_query, k=10)

    # Combine scores from both retrievals
    combined_scores = {}
    for rank, idx in enumerate(dense_indices):
        combined_scores[idx] = combined_scores.get(idx, 0) + dense_weight * (1 / (1 + rank))
    for rank, idx in enumerate(sparse_indices):
        combined_scores[idx] = combined_scores.get(idx, 0) + sparse_weight * (1 / (1 + rank))

    # Boost scores if genres match
    if boost_genre and detected_genre:
        for i, title in enumerate(titles):
            genre_text = title_to_text.get(title, "").lower()
            if "genre: unknown" in genre_text:
                combined_scores[i] = combined_scores.get(i, 0) - 0.75
            for g in detected_genre:
                if g in genre_text:
                    combined_scores[i] = combined_scores.get(i, 0) + 1.0
                    break

    # Boost if movie title appears in query
    if boost_title:
        query_lower = query.lower()
        for i, title in enumerate(titles):
            if title.lower().split("_")[0] in query_lower:
                combined_scores[i] = combined_scores.get(i, 0) + 2.0

    # Sort final scores
    sorted_indices = sorted(combined_scores, key=combined_scores.get, reverse=True)

    # Filter to keep only movies with matching or unknown genre
    if detected_genre:
        genre_filtered_indices = []
        for i in sorted_indices:
            genre_section = title_to_text.get(titles[i], "").lower()
            if "genre: unknown" in genre_section or any(g in genre_section for g in detected_genre):
                genre_filtered_indices.append(i)
        if genre_filtered_indices:
            sorted_indices = genre_filtered_indices[:k]

    # Final top results
    top_titles = [titles[i] for i in sorted_indices[:k]]
    top_docs = [title_to_text[t] for t in top_titles]

    return top_docs, top_titles


In [26]:
query = "Suggest a scifi movie, in which aliens come to Earth"
top_docs, top_titles = hybrid_retrieve(query)

print(f"\nTop results for: {query}")
for title, doc in zip(top_titles, top_docs):
    print(f"🎬 {title}")
    print(doc[:300] + "...\n")

Detected genres: ['science fiction']
Extracted keywords: ['scifi', 'alien', 'earth']

Top results for: Suggest a scifi movie, in which aliens come to Earth
🎬 The Day the Earth Stood Still_2008
Movie: The Day the Earth Stood Still Origin: American Genre: science fiction Director: Scott Derrickson Cast: Keanu Reeves, Jennifer Connelly, Jaden Smith, Kathy Bates Plot: In 1928, a solitary mountaineer encounters a glowing sphere. He loses consciousness and when he wakes, the sphere has gone and...

🎬 Men in Black_1997
Movie: Men in Black Origin: American Genre: science fiction Director: Barry Sonnenfeld Cast: Will Smith, Tommy Lee Jones, Vincent D'Onofrio, Linda Fiorentino, Rip Torn, Tony Shalhoub Plot: After a government agency makes first contact with aliens in 1961, alien refugees live in secret on Earth, most...

🎬 Alien_1979
Movie: Alien Origin: American Genre: unknown Director: Ridley Scott Cast: Sigourney Weaver, Tom Skerritt, Harry Dean Stanton, Veronica Cartwright, John Hurt, Ian Ho

## Generation System using T5

In [27]:
# rm -rf ~/.cache/huggingface/hub/models--google--flan-t5-base

In [28]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "google/flan-t5-base"
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)

In [29]:
# input_text = "translate English to Gernman: How old are you?"
# input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# outputs = model.generate(input_ids)
# print(tokenizer.decode(outputs[0]))

In [30]:
def generate_answer(query, model=model, tokenizer=tokenizer, max_input_len=512, max_output_len=300):
    context = "\n\n".join([f"Movie: {title}\n{plot}" for title, plot in zip(top_titles[:1], top_docs[:1])])
    prompt = (
        f"You are an expert in movies.\n"
        f"Task: Why is this a good movie recommendation for question: {query}\n\n"
        f"Context: {context}\n\n"
        f"Answer in one full sentence:"
    )
    print(f"User: {query}")

    input_ids = tokenizer(prompt, return_tensors="pt", max_length=max_input_len, truncation=False).input_ids
    outputs = model.generate(input_ids, max_new_tokens=max_output_len, num_beams=4, early_stopping=True, no_repeat_ngram_size=2)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

In [31]:
answer = generate_answer(query)
print(f"Movie bot: {answer}")

User: Suggest a scifi movie, in which aliens come to Earth
Movie bot: The Day the Earth Stood Still
